In [1]:
import torch
import torch.nn 
import torch.nn.functional
from typing import Union, List

batch_size = 64
context_len = 256

device = "cuda" if torch.cuda.is_available() else "cpu"

# Architettura e Implementazione di un Transformer (Decoder-only)

## 1. Rappresentazione del Dato e Pre-processing
L'obiettivo primario è la mappatura di un corpus testuale in uno spazio vettoriale computabile dal modello.

* **Tokenizzazione a livello di Carattere**: Identificazione del set di caratteri univoci (vocabolario) per definire la dimensionalità dello spazio di input.
* **Encoding Bidirezionale**: Creazione di funzioni di mapping (`token_to_id` e `id_to_token`) per tradurre i token in indici interi e viceversa.
* **Configurazione degli Iperparametri**: Definizione della dimensione dei batch (parallelismo) e della lunghezza del contesto (finestra di attenzione), parametri che condizionano la memoria e la capacità computazionale.
* **Suddivisione del Dataset**: Partizionamento tra training e validation set per garantire una corretta valutazione della capacità di generalizzazione.

PS: Nella `token_to_it` usa `"".join`

In [6]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data

    #select a starting point
    ix = torch.randint(len(data) - context_len, (batch_size,))

    #select starting point + context_len
    x = torch.stack([data[i:i+context_len] for i in ix])

    #select target
    y = torch.stack([data[i+1:i+context_len+1] for i in ix])

    x, y = x.to(device), y.to(device)
    return x, y

In [7]:
get_batch("train")

(tensor([[43,  1, 53,  ..., 58, 53, 56],
         [58, 43, 52,  ...,  1, 39, 45],
         [ 1, 52, 53,  ..., 59, 52, 57],
         ...,
         [ 0, 32, 46,  ..., 42, 56, 63],
         [58,  1, 51,  ..., 58, 43, 42],
         [ 1, 55, 59,  ..., 46, 43, 39]], device='cuda:0'),
 tensor([[ 1, 53,  5,  ..., 53, 56,  8],
         [43, 52,  1,  ..., 39, 45, 39],
         [52, 53, 58,  ..., 52, 57, 43],
         ...,
         [32, 46, 43,  ..., 56, 63,  1],
         [ 1, 51, 43,  ..., 43, 42,  1],
         [55, 59, 43,  ..., 43, 39, 42]], device='cuda:0'))

## 3. Scaled Dot-Product Attention
Il cuore del Transformer risiede nella gestione della focalizzazione del modello sulle parti rilevanti della sequenza tramite la classe `Head`.

$$
 \text{Attention}(Q, K, V) = \text{Softmax}\left(\frac{QK^T}{\sqrt{d_k}} \right)V
$$

* **Proiezioni Lineari (Q, K, V)**: Trasformazione dell'input in vettori *Query*, *Key* e *Value* tramite matrici di peso dedicate.
* **Calcolo della Matrice di Affinità**: Prodotto scalare tra $Q$ e $K$, normalizzato per la radice quadrata della dimensione della testa per mantenere la stabilità dei gradienti.
* **Causal Masking**: Applicazione di una maschera triangolare inferiore per forzare l'attenzione a ignorare i token futuri, rispettando il vincolo di causalità del decoder.
* **Distribuzione di Probabilità**: Utilizzo della funzione Softmax per pesare l'influenza dei diversi token sulla rappresentazione finale.

## 4. Modularità e Composizione: Il Transformer Block
Integrazione di componenti modulari per facilitare l'apprendimento di feature complesse.

* **Multi-Head Attention**: Implementazione di più teste di attenzione in parallelo (es. 6 teste) per catturare simultaneamente diverse relazioni semantiche.
* **Feed-Forward Network (FFN)**: Stadio di elaborazione locale che applica trasformazioni lineari e non lineari (ReLU) ad ogni posizione della sequenza indipendentemente.
* **Stabilità Numerica**: Integrazione di *Layer Normalization* e connessioni residue (*Residual Connections*) per mitigare il problema della scomparsa del gradiente in architetture profonde.

## 5. Architettura Globale e Generazione
Integrazione finale dei moduli per la creazione del modello completo.

* **Embedding Combinato**: Somma di *Token Embedding* e *Positional Embedding* per fornire al modello sia l'identità del carattere che la sua coordinata spaziale nella sequenza.
* **Deep Stacking**: Utilizzo di una sequenza di blocchi Transformer (es. 8 blocchi) per incrementare la profondità della rappresentazione.
* **Output Head**: Proiezione lineare finale verso lo spazio del vocabolario per la generazione dei logit.
* **Inference Autoregressiva**: Generazione di nuovo testo partendo da un seed iniziale, campionando iterativamente dalla distribuzione di probabilità predetta.

PS: Ricorda dropout in HEAD nel prodotto qk e in multi head, `p = 0.2 `

PS: Module list per la multi head, poi torch.cat

PS: Ricorda la projection alla fine di tutto per riportare da emb_len a n_tokens

## Transoformers stuff



In [ ]:
class Head(torch.nn.Module):
    def __init__(self, embedding_len, head_size):
        super().__init__()

        self.register_buffer("tril", torch.tril(torch.ones(context_len, context_len)))
        self.dropout = torch.nn.Dropout(p=0.2)
        self.sm = torch.nn.Softmax(dim=-1)
    
    def forward(self, x):
        # input x [batch_size, context_len, embedding_len]
        return qk @ v

class MultiHeadAttention(torch.nn.Module):
    def __init__(self, embedding_len, num_heads, head_size):
        super().__init__()
    def forward(self,x): 
        return out


class FeedForward(torch.nn.Module):
    # ff part of the transformer
    def __init__(self, embedding_len):
        super().__init__()


    def forward(self,x):
 
        return x

class Block(torch.nn.Module):
    def __init__(self, embedding_len, num_heads):
        super().__init__()
        assert embedding_len % num_heads == 0, "For this example embedding len and num_heads should be multiple"

    def forward(self,x):

        return x


class PicoGPT(torch.nn.Module):
    def __init__(self, embedding_len, num_heads, num_blocks, vocab_size, context_len):
        # embeddings and positional embeddings
        super().__init__()

        # use unpack operator to get the thing unpacked in the args

    
    def forward(self,data):
        # retrieve the context len

        return self.projection(self.ln(x))


        

In [13]:
num_heads = 6
embedding_len = 384
num_blocks = 8
num_iters = 1000

model = PicoGPT(embedding_len=embedding_len, num_heads=num_heads, num_blocks=num_blocks, vocab_size=num_tokens, context_len=context_len)
loss = torch.nn.CrossEntropyLoss()
optim = torch.optim.AdamW(model.parameters(), lr = 3e-4)



In [ ]:
model.to(device)
model.train()
every = 100
loss_vals = [0. in range(num_iters)]
for i in range(num_iters):
    # training loop
  
    if i% every == 0: print(f"Loss @ iteration {i} : {loss_val}")

Loss @ iteration 0 : 4.356752395629883


KeyboardInterrupt: 

In [12]:
#generation step

seq = "a"
seq_token = torch.tensor(encode_id(seq)).unsqueeze(1)

In [14]:
seq_token
sf = torch.nn.Softmax(dim = -1)
model.eval()
for i in range(len(seq_token) - 1, 200):
    logits = model(seq_token)
    next_token = torch.multinomial(sf(logits[:,-1,:]), 1).item()
    seq_token = torch.cat((seq_token, torch.tensor(next_token).reshape(1,1)), dim=1)
    #print(decode_id(seq_token[0].numpy()))


In [15]:
seq_token.shape

torch.Size([1, 26])

In [15]:

print(decode_id(seq_token[0].numpy()))

angru hy yollon ane's Bfourecphuthin ve t wiighede pad lll neshedsserd t and and,
Th Bun ti an:
Thooo t forn:
RDYa s akivee helofisthteveme ber rsthe, Coiverest, fofacke arid s stwls eniry mur whereared t Fr'seatMFrce wind ke 
